# 082 — Dimensionar hardware: de la laptop al clúster

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** FP32 **52 GB**, BF16 **26 GB**, INT8 **13 GB**, Q4_K_M **7,4 GB**.
Con 4 GB reservados quedan 20 GB útiles: entran INT8 y Q4; BF16 y FP32 no. Nota
que el salto BF16 → INT8 es el que cambia la decisión de compra, no el de INT8 → Q4.

**Ejercicio 2.** KV/token = 2 × 32 × 8 × 128 × 2 = **131 072 B ≈ 0,131 MB**.
Total = 0,131 MB × 8 192 × 16 ≈ **17,2 GB**. Sumando: 4,6 + 17,2 + 2 = **23,8 GB**
frente a 24 GB de VRAM. Técnicamente "cabe" con 0,2 GB de margen, es decir: **no
cabe**. Cualquier pico lo tumba. El KV es 3,7× los pesos — dimensionar sólo por el
modelo habría dado la respuesta contraria.

**Ejercicio 3.** Fine-tuning completo: 8e9 × 16 B = **128 GB** → **2 H100**
(y en la práctica más, por activaciones y fragmentación). QLoRA: 8e9 × 0,5 ≈ 4 GB
más ~2 GB ≈ **6 GB** → cabe de sobra en **1 GPU de consumo**. La diferencia no es
de grado: cambia quién puede entrenar.

**Ejercicio 4.** Porque la respuesta depende de la carga real —distribución de
longitudes, concurrencia pico, objetivo de p99— y ésas se estiman con datos y se
verifican con evals, no con una división. Dimensionar sin medir produce números
exactos y equivocados.


In [ ]:
# Ejercicio 1
n = 13e9
for fmt, b in {"FP32": 4, "BF16": 2, "INT8": 1, "Q4_K_M": 0.57}.items():
    gb = n * b / 1e9
    print(f"{fmt:8} {gb:6.1f} GB  {'entra' if gb <= 20 else 'NO entra'} en 24 GB - 4 GB")

# Ejercicio 2
kv_token = 2 * 32 * 8 * 128 * 2
kv_total = kv_token * 8192 * 16 / 1e9
total = 4.6 + kv_total + 2
print(f"KV/token={kv_token} B  KV total={kv_total:.1f} GB  total={total:.1f} GB de 24 GB")

# Ejercicio 3
full_ft = 8e9 * 16 / 1e9
qlora = 8e9 * 0.5 / 1e9 + 2
print(f"full FT={full_ft:.0f} GB → {-(-full_ft // 80):.0f} H100 · QLoRA={qlora:.0f} GB → 1 GPU")

# Ejercicio 4
result = run_lab("evaluation", seed=82)
assert result["kind"] == "evaluation" and result["seed"] == 82
assert result["evidence"] and result["limitations"]
show(result)


## Reflexión

1. En el ejercicio 2 el KV cache triplica a los pesos. ¿Qué tres palancas tienes
   para que quepa, y cuál degrada menos el producto?
2. Tu equipo pide "una GPU más grande" porque el modelo no entra. ¿Qué preguntas
   harías antes de aprobar la compra?
3. ¿En qué condiciones una máquina con memoria unificada de 192 GB es mejor
   elección que dos GPUs de 48 GB, y en cuáles es claramente peor?
